# BirdCLEF+ 2026 — Week 2 Training (Optimised)

## Key change vs previous run
Mel spectrograms are pre-computed once and saved as `.npy` files before training begins.
This drops epoch time from **33 min → ~4 min**, fitting 3 folds × 10 epochs in **~3 hours**.

| Step | Time estimate |
|---|---|
| Pre-compute 35k spectrograms (4 threads) | ~40–60 min |
| Train fold 0–2 × 10 epochs | ~120 min |
| **Total** | **~3 hours** |

**Settings:** Internet ON · GPU T4 · Accelerator enabled

In [ ]:
import os, gc, math, random, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device :', DEVICE)
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
class CFG:
    # Paths
    BASE_DIR        = Path('/kaggle/input/competitions/birdclef-2026')
    TRAIN_AUDIO_DIR = BASE_DIR / 'train_audio'
    TRAIN_CSV       = BASE_DIR / 'train.csv'
    SAMPLE_SUB      = BASE_DIR / 'sample_submission.csv'
    OUTPUT_DIR      = Path('/kaggle/working')
    CACHE_DIR       = Path('/kaggle/working/spec_cache')  # pre-computed .npy files

    # Audio
    SAMPLE_RATE     = 32000
    WINDOW_SIZE     = 5

    # Mel spectrogram — identical to Week 1 and inference notebook
    N_FFT           = 1024
    HOP_LENGTH      = 64
    N_MELS          = 136
    FMIN            = 20
    FMAX            = 16000
    TARGET_SHAPE    = (256, 256)

    # Model
    MODEL_NAME      = 'efficientnet_b0'
    WEIGHTS_PATH    = None  # None = download via timm (internet ON)

    # Training
    N_FOLDS         = 5
    TRAIN_FOLDS     = [0, 1, 2]   # 3 folds — fits in ~3 hours with pre-computation
    EPOCHS          = 10           # enough with 3-fold ensemble
    BATCH_SIZE      = 64
    NUM_WORKERS     = 2            # safe for .npy loading; prefetching helps GPU utilisation
    LR              = 1e-3
    WEIGHT_DECAY    = 1e-4
    WARMUP_EPOCHS   = 1
    MIN_LR          = 1e-6
    MIXUP_ALPHA     = 0.15

    # SpecAugment
    FREQ_MASK_MAX   = 30
    TIME_MASK_MAX   = 60
    N_FREQ_MASKS    = 2
    N_TIME_MASKS    = 2

    # Pre-computation
    PRECOMPUTE_WORKERS = 4   # threads for parallel spectrogram pre-computation

    SEED            = 42
    DEBUG           = False

cfg = CFG()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir :', cfg.OUTPUT_DIR)
print('Cache dir  :', cfg.CACHE_DIR)

In [ ]:
# Load training metadata
train_df = pd.read_csv(cfg.TRAIN_CSV)
species_col = 'primary_label' if 'primary_label' in train_df.columns else 'species_code'
print(f'Train rows   : {len(train_df)} | species col: "{species_col}"')

all_classes = sorted(train_df[species_col].unique())
NUM_CLASSES = len(all_classes)
le = LabelEncoder()
le.fit(all_classes)
class_list = le.classes_.tolist()
print(f'Classes      : {NUM_CLASSES}')

train_df['label_idx'] = le.transform(train_df[species_col])

if cfg.DEBUG:
    train_df = train_df.groupby(species_col).head(2).reset_index(drop=True)
    print(f'DEBUG: using {len(train_df)} rows')

print(f'First 3 classes: {class_list[:3]}')

In [ ]:
# Audio and spectrogram functions — identical to Week 1 and inference notebook
def load_audio(filepath, sr, offset=0.0, duration=5.0):
    """Load audio, tile short clips instead of zero-padding."""
    target = int(sr * duration)
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=offset, duration=duration, mono=True)
    except Exception:
        return np.zeros(target, dtype=np.float32)
    if len(y) == 0:
        return np.zeros(target, dtype=np.float32)
    if len(y) < target:
        reps = math.ceil(target / len(y))
        y = np.tile(y, reps)
    return y[:target].astype(np.float32)


def to_melspec(y, cfg):
    mel = librosa.feature.melspectrogram(
        y=y, sr=cfg.SAMPLE_RATE, n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH,
        n_mels=cfg.N_MELS, fmin=cfg.FMIN, fmax=cfg.FMAX, power=2.0)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
    return mel.astype(np.float32)


def spec_augment(spec, cfg):
    spec = spec.copy()
    H, W = spec.shape
    for _ in range(cfg.N_FREQ_MASKS):
        f  = random.randint(0, cfg.FREQ_MASK_MAX)
        f0 = random.randint(0, max(0, H - f))
        spec[f0:f0+f, :] = 0.0
    for _ in range(cfg.N_TIME_MASKS):
        t  = random.randint(0, cfg.TIME_MASK_MAX)
        t0 = random.randint(0, max(0, W - t))
        spec[:, t0:t0+t] = 0.0
    return spec


def mixup(x, y, alpha=0.15):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam


def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1-lam) * criterion(pred, yb)


print('Audio pipeline defined.')

## Pre-compute Mel Spectrograms

Each audio file is loaded once, converted to a mel spectrogram, and saved as a float16 `.npy` file.
Subsequent epochs load from this cache in ~2ms instead of ~400ms — a **200× speedup** per sample.

Storage: 35,000 files × 256×256 × 2 bytes ≈ **4.6 GB** (well within Kaggle's 20 GB working dir).

In [ ]:
def cache_key(filename):
    """Convert 'species/XC123456.ogg' → 'species_XC123456.ogg.npy'"""
    return filename.replace('/', '_').replace('\\', '_') + '.npy'


def precompute_one(args):
    """Compute and save one spectrogram. Skips if already cached."""
    filename, audio_dir, cache_dir, cfg = args
    dst = cache_dir / cache_key(filename)
    if dst.exists():
        return 'skip'
    try:
        fp = str(audio_dir / filename)
        y  = load_audio(fp, cfg.SAMPLE_RATE, 0.0, cfg.WINDOW_SIZE)
        spec = to_melspec(y, cfg)
        spec = cv2.resize(spec, (cfg.TARGET_SHAPE[1], cfg.TARGET_SHAPE[0]),
                          interpolation=cv2.INTER_CUBIC)
        np.save(str(dst), spec.astype(np.float16))  # float16 halves storage
        return 'ok'
    except Exception as e:
        return f'err:{e}'


# Build task list
tasks = [
    (row['filename'], cfg.TRAIN_AUDIO_DIR, cfg.CACHE_DIR, cfg)
    for _, row in train_df.iterrows()
]

already_cached = sum(1 for _, row in train_df.iterrows()
                     if (cfg.CACHE_DIR / cache_key(row['filename'])).exists())
print(f'Files to process : {len(tasks)}')
print(f'Already cached   : {already_cached}')
print(f'Need to compute  : {len(tasks) - already_cached}')
print(f'Using {cfg.PRECOMPUTE_WORKERS} threads — this takes ~40–60 min for a fresh run')
print(f'Storage needed   : ~{len(tasks) * 256*256*2 / 1e9:.1f} GB')

In [ ]:
import time as _time
_t0 = _time.time()

counts = {'skip': 0, 'ok': 0, 'err': 0}

with ThreadPoolExecutor(max_workers=cfg.PRECOMPUTE_WORKERS) as pool:
    futures = {pool.submit(precompute_one, t): t[0] for t in tasks}
    with tqdm(total=len(futures), desc='Pre-computing spectrograms') as pbar:
        for fut in as_completed(futures):
            result = fut.result()
            key = result if result in counts else 'err'
            counts[key] += 1
            pbar.update(1)
            if counts['ok'] % 1000 == 1:
                elapsed = _time.time() - _t0
                rate = (counts['ok'] + counts['skip']) / elapsed
                remaining = (len(tasks) - counts['ok'] - counts['skip']) / max(rate, 1)
                pbar.set_postfix(ok=counts['ok'], skip=counts['skip'],
                                 eta=f'{remaining/60:.0f}min')

print(f'\nDone in {(_time.time()-_t0)/60:.1f} min')
print(f'  Computed : {counts["ok"]}')
print(f'  Skipped  : {counts["skip"]} (already cached)')
print(f'  Errors   : {counts["err"]}')

# Verify cache
cached_count = sum(1 for _, row in train_df.iterrows()
                   if (cfg.CACHE_DIR / cache_key(row['filename'])).exists())
print(f'\nCache coverage: {cached_count}/{len(train_df)} files ({100*cached_count/len(train_df):.1f}%)')

In [ ]:
class BirdDataset(Dataset):
    """
    Fast dataset that loads pre-computed mel spectrograms from .npy cache.
    Falls back to on-the-fly computation if a file is not cached.
    """
    def __init__(self, df, cfg, augment=True):
        self.df      = df.reset_index(drop=True)
        self.cfg     = cfg
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']

        # Load from cache (fast path)
        cache_path = self.cfg.CACHE_DIR / cache_key(filename)
        if cache_path.exists():
            spec = np.load(str(cache_path)).astype(np.float32)
        else:
            # Fallback: compute on-the-fly (slow, only if cache miss)
            fp   = str(self.cfg.TRAIN_AUDIO_DIR / filename)
            y    = load_audio(fp, self.cfg.SAMPLE_RATE, 0.0, self.cfg.WINDOW_SIZE)
            spec = to_melspec(y, self.cfg)
            spec = cv2.resize(spec, (self.cfg.TARGET_SHAPE[1], self.cfg.TARGET_SHAPE[0]),
                              interpolation=cv2.INTER_CUBIC)

        # Augmentation applied at runtime (not cached — different each epoch)
        if self.augment:
            spec = spec_augment(spec, self.cfg)

        tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)  # (1, H, W)

        # Label vector
        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        label[int(row['label_idx'])] = 1.0

        # Secondary labels at 0.5 weight
        sec = str(row.get('secondary_labels', ''))
        if sec and sec not in ('nan', 'None', ''):
            for sp in sec.replace(',', ' ').split():
                sp = sp.strip()
                if sp in le.classes_:
                    label[le.transform([sp])[0]] = 0.5

        return tensor, label


print('BirdDataset defined (cache-backed).')

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(self.eps).pow(self.p), 1).pow(1/self.p)


class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes, cfg):
        super().__init__()
        if cfg.WEIGHTS_PATH and Path(cfg.WEIGHTS_PATH).exists():
            self.backbone = timm.create_model(
                cfg.MODEL_NAME, pretrained=False,
                num_classes=0, global_pool='', in_chans=1)
            sd  = torch.load(cfg.WEIGHTS_PATH, map_location='cpu', weights_only=True)
            key = 'conv_stem.weight'
            if key in sd:
                sd[key] = sd[key].mean(dim=1, keepdim=True)
            self.backbone.load_state_dict(sd, strict=False)
            print(f'Loaded weights from {cfg.WEIGHTS_PATH}')
        else:
            self.backbone = timm.create_model(
                cfg.MODEL_NAME, pretrained=True,
                num_classes=0, global_pool='', in_chans=1)
            print('Pretrained weights downloaded via timm.')

        d = self.backbone.num_features
        self.pool = GeM()
        self.bn   = nn.BatchNorm1d(d)
        self.drop = nn.Dropout(p=0.3)
        self.fc   = nn.Linear(d, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.drop(x)
        return self.fc(x)


# Quick test
_m = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
_o = _m(torch.randn(2, 1, 256, 256).to(DEVICE))
print('Output shape:', _o.shape)
print(f'Parameters  : {sum(p.numel() for p in _m.parameters())/1e6:.1f}M')
del _m, _o; gc.collect(); torch.cuda.empty_cache()

In [ ]:
def get_scheduler(optimizer, cfg, steps_per_epoch):
    total  = cfg.EPOCHS * steps_per_epoch
    warmup = cfg.WARMUP_EPOCHS * steps_per_epoch
    def lr_fn(step):
        if step < warmup:
            return step / max(1, warmup)
        prog = (step - warmup) / max(1, total - warmup)
        return max(cfg.MIN_LR / cfg.LR, 0.5 * (1 + math.cos(math.pi * prog)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


def compute_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            try: aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
            except: pass
    return float(np.mean(aucs)) if aucs else 0.0


def train_epoch(model, loader, optimizer, scheduler, criterion, cfg):
    model.train()
    losses = []
    for x, y in tqdm(loader, desc='  Train', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        if cfg.MIXUP_ALPHA > 0 and random.random() > 0.5:
            x, ya, yb, lam = mixup(x, y, cfg.MIXUP_ALPHA)
            loss = mixup_loss(criterion, model(x), ya, yb, lam)
        else:
            loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc='  Valid', leave=False):
        preds.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
        labels.append(y.numpy())
    preds  = np.concatenate(preds)
    labels = np.concatenate(labels)
    return compute_auc((labels > 0.5).astype(int), preds), preds, labels


print('Training utilities defined.')

In [ ]:
# Stratified K-Fold split
skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['label_idx'])):
    train_df.loc[train_df.index[val_idx], 'fold'] = fold

print('Fold distribution:')
print(train_df['fold'].value_counts().sort_index().to_string())

In [ ]:
oof_preds  = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
oof_labels = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
best_aucs  = {}

criterion = nn.BCEWithLogitsLoss()

for fold in cfg.TRAIN_FOLDS:
    print(f'\n{"="*55}')
    print(f'  FOLD {fold}')
    print(f'{"="*55}')

    trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f'  Train: {len(trn_df)}  |  Val: {len(val_df)}')

    trn_ds = BirdDataset(trn_df, cfg, augment=True)
    val_ds = BirdDataset(val_df, cfg, augment=False)

    trn_loader = DataLoader(trn_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                            num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                            num_workers=cfg.NUM_WORKERS, pin_memory=True)

    model     = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, cfg, len(trn_loader))

    best_auc  = 0.0
    best_path = cfg.OUTPUT_DIR / f'model_fold{fold}.pt'
    history   = []

    for epoch in range(1, cfg.EPOCHS + 1):
        trn_loss         = train_epoch(model, trn_loader, optimizer, scheduler, criterion, cfg)
        val_auc, preds, labels = validate(model, val_loader)
        history.append({'epoch': epoch, 'loss': trn_loss, 'val_auc': val_auc})

        lr_now = scheduler.get_last_lr()[0]
        marker = ' ✔' if val_auc > best_auc else ''
        print(f'  E{epoch:02d}  loss={trn_loss:.4f}  auc={val_auc:.4f}  lr={lr_now:.2e}{marker}')

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)

    best_aucs[fold] = best_auc
    print(f'\n  Fold {fold} best AUC: {best_auc:.4f}')

    # OOF predictions
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    val_idx_orig = train_df[train_df['fold'] == fold].index
    _, oof_p, oof_l = validate(model, val_loader)
    oof_preds[val_idx_orig]  = oof_p
    oof_labels[val_idx_orig] = (oof_l > 0.5).astype(np.float32)

    # Training curve
    hist = pd.DataFrame(history)
    fig, ax1 = plt.subplots(figsize=(9, 3))
    ax1.plot(hist['epoch'], hist['loss'], 'b-o', ms=4)
    ax1.set_ylabel('BCE Loss', color='b')
    ax2 = ax1.twinx()
    ax2.plot(hist['epoch'], hist['val_auc'], 'r-s', ms=4)
    ax2.set_ylabel('Val AUC', color='r')
    ax1.set_xlabel('Epoch')
    plt.title(f'Fold {fold}  —  best AUC {best_auc:.4f}')
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / f'curve_fold{fold}.png', dpi=100)
    plt.show()

    del model, trn_loader, val_loader, trn_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()

print(f'\nAll folds done.')
print('Best AUCs:', {k: round(v, 4) for k, v in best_aucs.items()})
print(f'Mean AUC : {np.mean(list(best_aucs.values())):.4f}')

In [ ]:
# OOF summary — only trained folds
trained_idx = train_df[train_df['fold'].isin(cfg.TRAIN_FOLDS)].index
oof_auc = compute_auc(oof_labels[trained_idx], oof_preds[trained_idx])
print(f'OOF AUC (folds {cfg.TRAIN_FOLDS}): {oof_auc:.4f}')

per_class = []
for i in range(NUM_CLASSES):
    t, p = oof_labels[trained_idx, i], oof_preds[trained_idx, i]
    if t.sum() > 0:
        try: per_class.append(roc_auc_score(t, p))
        except: pass

plt.figure(figsize=(8, 3))
plt.hist(per_class, bins=30, color='steelblue', edgecolor='white')
plt.axvline(np.mean(per_class), color='red', linestyle='--',
            label=f'Mean = {np.mean(per_class):.3f}')
plt.xlabel('Per-class ROC-AUC'); plt.title('OOF per-class AUC')
plt.legend(); plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'oof_auc.png', dpi=100)
plt.show()

print(f'Classes < 0.7 AUC: {sum(a < 0.7 for a in per_class)}')
print(f'Classes < 0.8 AUC: {sum(a < 0.8 for a in per_class)}')
print('\nSaved model files:')
for f in sorted(cfg.OUTPUT_DIR.glob('model_fold*.pt')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')

## After this notebook completes

1. **Save Version** → all `model_fold0.pt`, `model_fold1.pt`, `model_fold2.pt` are in the output
2. **Publish as a new Kaggle Dataset** (Output tab → New Dataset)
3. **Update inference notebook** — add all 3 model paths to `MODEL_PATHS`:

```python
MODEL_PATHS = [
    '/kaggle/input/YOUR-DATASET/model_fold0.pt',
    '/kaggle/input/YOUR-DATASET/model_fold1.pt',
    '/kaggle/input/YOUR-DATASET/model_fold2.pt',
]
```

4. **Submit** — the 3-model ensemble runs inference and averages predictions

### Expected public LB improvement
| Source | Gain |
|---|---|
| 3-fold ensemble vs 1-fold | +0.03 to +0.06 |
| Tiling short clips (not zero-padding) | +0.01 to +0.02 |
| 10 epochs with better convergence | +0.01 to +0.02 |
| **Expected total** | **~0.74 to 0.80** |